In [2]:

import pandas as pd
import re
import os
import copy

# STEP 0: LOAD ALL 14 DATASETS
PARQUET = "/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet"

hpa_rna         = pd.read_parquet(f"{PARQUET}/1_4_hpa_rna_celline.parquet")
depmap_expr     = pd.read_parquet(f"{PARQUET}/2_DepMap_OmicsExpressionAllGenesTPMLogp1Profile.parquet")
geo_expr        = pd.read_parquet(f"{PARQUET}/3_GEOexpression.parquet")
proteomics      = pd.read_parquet(f"{PARQUET}/4_Harmonized_MS_CCLE_Gygi_subsetted.parquet")
fusions         = pd.read_parquet(f"{PARQUET}/5_OmicsFusionFilteredSupplementary.parquet")
mutations       = pd.read_parquet(f"{PARQUET}/6_OmicsSomaticMutationsProfile.parquet")
cellosaurus     = pd.read_parquet(f"{PARQUET}/7_cellosaurus.parquet")
depmap_profiles = pd.read_parquet(f"{PARQUET}/8_DepMap_OmicsProfiles.parquet")
sample_info     = pd.read_parquet(f"{PARQUET}/9_DepMap_sample_info.parquet")
geo_info        = pd.read_parquet(f"{PARQUET}/10_GEOInfo.parquet")
hpa_desc        = pd.read_parquet(f"{PARQUET}/11_hpa_rna_celline_description.parquet")
metabolomics    = pd.read_parquet(f"{PARQUET}/12_CCLE_metabolomics_20190502.parquet")
mirna           = pd.read_parquet(f"{PARQUET}/13_CCLE_miRNA_20181103.parquet")
signatures      = pd.read_parquet(f"{PARQUET}/14_OmicsGlobalSignatures.parquet")

tables = {
    "hpa_rna": hpa_rna,
    "depmap_expr": depmap_expr,
    "geo_expr": geo_expr,
    "proteomics": proteomics,
    "fusions": fusions,
    "mutations": mutations,
    "cellosaurus": cellosaurus,
    "depmap_profiles": depmap_profiles,
    "sample_info": sample_info,
    "geo_info": geo_info,
    "hpa_desc": hpa_desc,
    "metabolomics": metabolomics,
    "mirna": mirna,
    "signatures": signatures,
}

# Keep an untouched copy for before/after comparisons
tables_raw = copy.deepcopy(tables)

In [3]:
# STEP 1: CLEANING FUNCTIONS

def clean_string_columns(df, columns=None):
    """
    Strip leading/trailing whitespace, collapse double spaces,
    and lowercase all string/object columns.
    """
    df = df.copy()
    if columns is None:
        columns = df.select_dtypes(include=["object", "str"]).columns

    for col in columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
            .str.lower()
        )
        df[col] = df[col].replace("nan", pd.NA)

    return df


def strip_ensembl_version(df, columns=None):
    """
    Remove Ensembl version suffixes like '.21', '.14', '.8' from gene/
    transcript/protein IDs, wherever they appear in a string
    (e.g. 'dlg1 (ensg00000075711.21)' -> 'dlg1 (ensg00000075711)').
    """
    df = df.copy()
    if columns is None:
        columns = df.select_dtypes(include=["object", "str"]).columns

    pattern = r"(ens[gtp]\d+)\.\d+"

    for col in columns:
        df[col] = df[col].astype(str).str.replace(
            pattern, r"\1", regex=True, flags=re.IGNORECASE
        )
        df[col] = df[col].replace("nan", pd.NA)

    return df


def transpose_table(df, id_col=None, new_index_name="sample_id"):
    """
    Transpose a wide-format table so samples become rows and
    genes/features become columns.

    Parameters:
        df (pd.DataFrame): input dataframe (features as rows)
        id_col (str): column containing feature IDs (e.g. 'Gene')
        new_index_name (str): name for the new sample-ID column after transpose
    """
    df = df.copy()
    if id_col is not None:
        df = df.set_index(id_col)

    transposed = df.T
    transposed.index.name = new_index_name
    transposed = transposed.reset_index()

    return transposed


In [4]:
# STEP 2: PRE-CLEANING CHECKS

def check_clean_summary(tables: dict):
    """Flags object columns with leading/trailing spaces, double spaces, or uppercase."""
    rows = []
    for name, df in tables.items():
        obj_cols = df.select_dtypes(include=["object", "str"]).columns
        dirty_cols = []
        for col in obj_cols:
            s = df[col].dropna().astype(str)
            has_issue = (
                s.str.contains(r"^\s|\s$", regex=True).any()
                or s.str.contains(r"\s{2,}", regex=True).any()
                or s.str.contains(r"[A-Z]", regex=True).any()
            )
            if has_issue:
                dirty_cols.append(col)
        rows.append({
            "table": name,
            "n_object_cols": len(obj_cols),
            "n_dirty_cols": len(dirty_cols),
            "dirty_cols": dirty_cols,
            "clean": len(dirty_cols) == 0
        })
    return pd.DataFrame(rows)


def check_ensembl_version_summary(tables: dict, n_examples=3):
    """Flags object columns containing Ensembl IDs with version suffixes."""
    pattern = r"ens[gtp]\d+\.\d+"
    rows = []
    for name, df in tables.items():
        obj_cols = df.select_dtypes(include=["object", "str"]).columns
        affected = {}
        for col in obj_cols:
            s = df[col].dropna().astype(str)
            mask = s.str.contains(pattern, regex=True, case=False)
            if mask.any():
                affected[col] = s[mask].head(n_examples).tolist()
        rows.append({
            "table": name,
            "n_affected_cols": len(affected),
            "affected_cols": list(affected.keys()),
            "examples": affected
        })
    return pd.DataFrame(rows)

print("PRE-CLEANING CHECKS")

pre_whitespace = check_clean_summary(tables_raw)
pre_ensembl    = check_ensembl_version_summary(tables_raw)

print("\nWhitespace/Case Summary (pre):")
print(pre_whitespace)

print("\nEnsembl Version Suffix Summary (pre):")
print(pre_ensembl)

PRE-CLEANING CHECKS

Whitespace/Case Summary (pre):
              table  n_object_cols  n_dirty_cols  \
0           hpa_rna              3             3   
1       depmap_expr              0             0   
2          geo_expr              1             1   
3        proteomics              1             1   
4           fusions             21            13   
5         mutations             44            34   
6       cellosaurus             17            17   
7   depmap_profiles              5             4   
8       sample_info             27            25   
9          geo_info             22            21   
10         hpa_desc              7             7   
11     metabolomics              2             2   
12            mirna              2             2   
13       signatures              5             5   

                                           dirty_cols  clean  
0                        [Gene, Gene name, Cell line]  False  
1                                        

In [5]:
# STEP 3: APPLY CLEANING TO ALL 14 TABLES

for name in tables:
    tables[name] = clean_string_columns(tables[name])
    tables[name] = strip_ensembl_version(tables[name])

# Unpack back to individual variable names
(hpa_rna, depmap_expr, geo_expr, proteomics, fusions, mutations,
 cellosaurus, depmap_profiles, sample_info, geo_info, hpa_desc,
 metabolomics, mirna, signatures) = tables.values()



In [6]:

# STEP 4: POST-CLEANING CHECKS

print("POST-CLEANING CHECKS")

post_whitespace = check_clean_summary(tables)
post_ensembl    = check_ensembl_version_summary(tables)

print("\nWhitespace/Case Summary (post — should all be clean=True):")
print(post_whitespace)

print("\nEnsembl Version Suffix Summary (post — should all be n_affected_cols=0):")
print(post_ensembl)

POST-CLEANING CHECKS

Whitespace/Case Summary (post — should all be clean=True):
              table  n_object_cols  n_dirty_cols dirty_cols  clean
0           hpa_rna              3             0         []   True
1       depmap_expr              0             0         []   True
2          geo_expr              1             0         []   True
3        proteomics              1             0         []   True
4           fusions             21             0         []   True
5         mutations             44             0         []   True
6       cellosaurus             17             0         []   True
7   depmap_profiles              5             0         []   True
8       sample_info             27             0         []   True
9          geo_info             22             0         []   True
10         hpa_desc              7             0         []   True
11     metabolomics              2             0         []   True
12            mirna              2             0

In [7]:
# ---------------------------------------------------------------
# STEP 5: BEFORE/AFTER COMPARISON (for worklog/dissertation evidence)
# ---------------------------------------------------------------

def combined_before_after(tables_before: dict, tables_after: dict, n_examples=2):
    """
    For each affected column, show example values before vs after cleaning.
    """
    rows = []
    ws_pattern  = re.compile(r"^\s|\s$|\s{2,}|[A-Z]")
    ens_pattern = re.compile(r"ens[gtp]\d+\.\d+", re.IGNORECASE)

    for name in tables_before:
        df_b = tables_before[name]
        df_a = tables_after[name]
        obj_cols = df_b.select_dtypes(include=["object", "str"]).columns

        for col in obj_cols:
            s_b = df_b[col].dropna().astype(str)
            ws_mask  = s_b.str.contains(ws_pattern, regex=True)
            ens_mask = s_b.str.contains(ens_pattern, regex=True)

            if ws_mask.any() or ens_mask.any():
                mask = ws_mask | ens_mask
                idx = s_b[mask].index[:n_examples]
                rows.append({
                    "table": name,
                    "column": col,
                    "issue_type": (
                        "whitespace/case" if ws_mask.any() and not ens_mask.any()
                        else "ensembl_version" if ens_mask.any() and not ws_mask.any()
                        else "both"
                    ),
                    "before": df_b.loc[idx, col].tolist(),
                    "after": df_a.loc[idx, col].tolist(),
                })
    return pd.DataFrame(rows)


comparison = combined_before_after(tables_raw, tables)
print("\nBefore/After Comparison (sample rows):")
print(comparison)


Before/After Comparison (sample rows):
          table                  column       issue_type  \
0       hpa_rna                    Gene  whitespace/case   
1       hpa_rna               Gene name  whitespace/case   
2       hpa_rna               Cell line  whitespace/case   
3      geo_expr                    Gene  whitespace/case   
4    proteomics              Unnamed: 0  whitespace/case   
..          ...                     ...              ...   
130  signatures            SequencingID  whitespace/case   
131  signatures                 ModelID  whitespace/case   
132  signatures        ModelConditionID  whitespace/case   
133  signatures  IsDefaultEntryForModel  whitespace/case   
134  signatures     IsDefaultEntryForMC  whitespace/case   

                                 before                               after  
0    [ENSG00000000003, ENSG00000000003]  [ensg00000000003, ensg00000000003]  
1                      [TSPAN6, TSPAN6]                    [tspan6, tspan6]  
2    

In [8]:
# ---------------------------------------------------------------
# STEP 6: TRANSPOSE GEO EXPRESSION (genes-as-rows -> samples-as-rows)
# ---------------------------------------------------------------
# depmap_expr (PR- ids as rows) and proteomics (ach- ids as rows)
# are already in samples-as-rows format -> no transpose needed.
#
# geo_expr is genes-as-rows, GSM samples as columns -> needs transposing.

geo_expr_T = transpose_table(geo_expr, id_col="Gene", new_index_name="sample_id")

print("TRANSPOSE CHECK")
print(f"geo_expr original shape: {geo_expr.shape}")
print(f"geo_expr_T shape:        {geo_expr_T.shape}")
print(geo_expr_T.head())

TRANSPOSE CHECK
geo_expr original shape: (19914, 3268)
geo_expr_T shape:        (3267, 19915)
Gene  sample_id  ensg00000000003  ensg00000000005  ensg00000000419  \
0     GSM101610        33.615700        40.925682      2182.281250   
1     GSM101615       553.249756        31.327406      3419.430420   
2     GSM101616       540.452209        33.934967      3514.540039   
3     GSM101667       599.431152        34.213123      2295.817383   
4     GSM101668       625.242737        32.466286      2378.469727   

Gene  ensg00000000457  ensg00000000460  ensg00000000938  ensg00000000971  \
0           58.934814       136.418900        58.961063        49.317619   
1           95.068222       257.929169        58.337406       111.684074   
2           94.900459       271.317230        64.827354       110.419968   
3           51.810070       162.090073        55.533710        79.690567   
4           52.327530       160.913849        51.006062        66.069588   

Gene  ensg00000001036  ensg0

In [9]:
# PIPELINE COMPLETE
print("\nPipeline complete.")
print("Cleaned tables available in `tables` dict and as individual variables.")
print("geo_expr_T is the transposed (samples-as-rows) version of GEO expression data.")


Pipeline complete.
Cleaned tables available in `tables` dict and as individual variables.
geo_expr_T is the transposed (samples-as-rows) version of GEO expression data.


In [10]:
"""
=====================================================================
OVERLAP CHECK — post-cleaning
Check ID overlaps across the 14 tables (gene IDs and cell line/sample IDs)
=====================================================================
"""

# ---------------------------------------------------------------
# STEP 1: Extract gene ID sets from each RNA/expression-like table
# ---------------------------------------------------------------

def extract_ensembl_ids(series):
    """Extract ensg IDs (cleaned, no version) from a column of strings."""
    pattern = r"ens[gtp]\d+"
    ids = series.dropna().astype(str).str.extract(f"({pattern})", flags=re.IGNORECASE)[0]
    return set(ids.dropna().str.lower().unique())


# depmap_expr columns look like "TSPAN6 (ENSG00000000003)"
depmap_genes = extract_ensembl_ids(pd.Series(depmap_expr.columns))

# geo_expr_T columns (after transpose) are gene IDs directly, e.g. "ensg00000000003"
geo_genes = set(
    str(c).lower() for c in geo_expr_T.columns
    if re.match(r"ens[gtp]\d+", str(c), flags=re.IGNORECASE)
)

# hpa_rna — check its columns/structure for gene ID column
print("hpa_rna columns sample:", list(hpa_rna.columns[:10]))
print("hpa_rna shape:", hpa_rna.shape)

hpa_rna columns sample: ['Gene', 'Gene name', 'Cell line', 'TPM', 'pTPM', 'nTPM']
hpa_rna shape: (24315372, 6)


In [11]:
# ---------------------------------------------------------------
# STEP 2: GENE-LEVEL OVERLAP (DepMap vs GEO vs HPA)
# ---------------------------------------------------------------

def overlap_summary(set_dict: dict):
    """
    Pairwise + 3-way overlap summary for a dict of {name: set}.
    """
    rows = []
    names = list(set_dict.keys())

    for name, s in set_dict.items():
        rows.append({"set": name, "n_unique": len(s)})

    print("Set sizes:")
    print(pd.DataFrame(rows))
    print()

    # pairwise overlaps
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            a, b = names[i], names[j]
            inter = set_dict[a] & set_dict[b]
            print(f"{a} ∩ {b}: {len(inter)} "
                  f"({len(inter) / len(set_dict[a]) * 100:.1f}% of {a}, "
                  f"{len(inter) / len(set_dict[b]) * 100:.1f}% of {b})")

    # 3-way overlap (if exactly 3 sets)
    if len(names) == 3:
        all_three = set_dict[names[0]] & set_dict[names[1]] & set_dict[names[2]]
        print(f"\n{names[0]} ∩ {names[1]} ∩ {names[2]}: {len(all_three)}")
        return all_three

    return None


gene_sets = {
    "depmap_expr": depmap_genes,
    "geo_expr": geo_genes,
    # "hpa_rna": hpa_genes,  # add once column identified
}

print("=" * 60)
print("GENE-LEVEL OVERLAP")
print("=" * 60)
common_genes = overlap_summary(gene_sets)

GENE-LEVEL OVERLAP
Set sizes:
           set  n_unique
0  depmap_expr     53961
1     geo_expr     19914

depmap_expr ∩ geo_expr: 19894 (36.9% of depmap_expr, 99.9% of geo_expr)


In [12]:
# ---------------------------------------------------------------
# STEP 3: SAMPLE / CELL LINE ID OVERLAP
# ---------------------------------------------------------------

# Inspect ID columns across key tables first
print("depmap_expr index sample:", depmap_expr.index[:5].tolist())
print("proteomics 'Unnamed: 0' sample:", proteomics["Unnamed: 0"].head().tolist() if "Unnamed: 0" in proteomics.columns else "check column name")
print("geo_expr_T sample_id sample:", geo_expr_T["sample_id"].head().tolist())
print("depmap_profiles columns:", list(depmap_profiles.columns))
print("sample_info columns:", list(sample_info.columns))
print("geo_info columns:", list(geo_info.columns))
print("cellosaurus columns:", list(cellosaurus.columns[:10]))

depmap_expr index sample: ['PR-AdBjpG', 'PR-I2AzwG', 'PR-5ekAAC', 'PR-I21681', 'PR-i9DRP1']
proteomics 'Unnamed: 0' sample: ['ach-000849', 'ach-000441', 'ach-000248', 'ach-000684', 'ach-000856']
geo_expr_T sample_id sample: ['GSM101610', 'GSM101615', 'GSM101616', 'GSM101667', 'GSM101668']
depmap_profiles columns: ['ProfileID', 'ModelCondition', 'ModelID', 'Datatype', 'WESKit']
sample_info columns: ['DepMap_ID', 'cell_line_name', 'stripped_cell_line_name', 'CCLE_Name', 'alias', 'COSMICID', 'sex', 'source', 'RRID', 'WTSI_Master_Cell_ID', 'sample_collection_site', 'primary_or_metastasis', 'primary_disease', 'Subtype', 'age', 'Sanger_Model_ID', 'depmap_public_comments', 'lineage', 'lineage_subtype', 'lineage_sub_subtype', 'lineage_molecular_subtype', 'default_growth_pattern', 'model_manipulation', 'model_manipulation_details', 'patient_id', 'parent_depmap_id', 'Cellosaurus_NCIt_disease', 'Cellosaurus_NCIt_id', 'Cellosaurus_issues']
geo_info columns: ['Geo_accession', 'CEL_file_names', 'tit

In [ ]:
"""
=====================================================================
TABLE HEADER INSPECTION — column names, dtypes, and sample values
across all 14 cleaned tables
=====================================================================
"""

def inspect_all_tables(tables: dict, n_samples=3):
    """
    For each table, print every column with its dtype and sample values.

    Parameters:
        tables (dict): dict of {name: DataFrame}
        n_samples (int): number of example values to show per column
    """
    for name, df in tables.items():
        print("=" * 70)
        print(f"TABLE: {name}   shape={df.shape}")
        print("=" * 70)
        print(f"{'#':<5} {'column':<40} {'dtype':<15} {'sample values'}")
        print("-" * 70)
        for i, col in enumerate(df.columns):
            dtype  = str(df[col].dtype)
            sample = df[col].dropna().head(n_samples).tolist()
            print(f"{i:<5} {str(col):<40} {dtype:<15} {sample}")
        print()


inspect_all_tables(tables, n_samples=3)

KeyError: 0